In [ ]:
import os
import socketio
import torch
from nnsight.intervention.backends.remote import RemoteBackend
from nnsight import LanguageModel


# Pick a model from the list of available models at https://host.edif.ai/status
model_name = "openai-community/gpt2"
model = LanguageModel(model_name)
tokenizer = model.tokenizer
API_KEY = os.getenv("EDIF_API_KEY", "")




# Create and patch RemoteBackend
remote_backend = RemoteBackend(
    model.to_model_key(),
    host="host.edif.ai/edif",
    ssl=True,
    api_key=API_KEY,
    blocking=True,
)

print("model.to_model_key():", model.to_model_key())
# Patch websocket path
original_connect = socketio.SimpleClient.connect
def patched_connect(self, *args, **kwargs):
    kwargs['socketio_path'] = "/edif/ws/socket.io"
    return original_connect(self, *args, **kwargs)

original_blocking_request = remote_backend.blocking_request
def patched_blocking_request(graph):
    socketio.SimpleClient.connect = patched_connect
    try:
        return original_blocking_request(graph)
    finally:
        socketio.SimpleClient.connect = original_connect

remote_backend.blocking_request = patched_blocking_request

d:\MAAI\MI\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model.to_model_key(): nnsight.modeling.language.LanguageModel:{"repo_id": "openai-community/gpt2", "revision": "main"}


In [ ]:
from sae_lens import SAE


release = "gpt2-small-attn-out-v5-128k"
sae_id = "blocks.0.hook_attn_out"
sae = SAE.from_pretrained(release, sae_id)


In [ ]:
# prompt = "The Eiffel Tower is located in"

# # Extract activations and compute SAE features
# with model.trace(prompt):
#     # This SAE is trained on GPT-2 block 0 attention output.
#     hidden_states = model.transformer.h[0].attn.output[0]

#     # Save the hidden states
#     hidden_states_saved = hidden_states.save()

# # Get SAE features outside the trace
# with torch.no_grad():
#     features = sae.encode(hidden_states_saved)

# print(f"Feature activations shape: {features.shape}")
# print(f"Average L0: {(features[:, 1:, :] > 0).sum(dim=-1).float().mean().item():.1f}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13169.21it/s]


Feature activations shape: torch.Size([1, 8, 131072])
Average L0: 32.0


In [11]:
def steer_with_feature_manual(prompt, feature_id, strength=5.0, max_new_tokens=15):
    """
    Steer generation by manually generating one token at a time,
    with intervention applied at each step.
    (Workaround for nnsight 0.5.3 generate() not respecting interventions)
    """
    delta = strength * sae.W_dec[feature_id].detach()
    
    # Initialize
    tokens = model.tokenizer.encode(prompt, return_tensors="pt")[0]  # [seq_len]
    result_tokens = tokens.clone()
    
    for i in range(max_new_tokens):
        # Run forward pass on current sequence
        with model.trace(model.tokenizer.decode(tokens)) as tracer:
            # Apply intervention at layer 6 (blocks.6.hook_resid_pre => h[5])
            h = model.transformer.h[5].output[0]
            h[:] = h + delta.unsqueeze(0).unsqueeze(0)
            
            # Get logits for next token
            logits = model.lm_head.output.save()
        
        # Sample next token (greedy: argmax)
        next_token = torch.argmax(logits[0, -1, :]).unsqueeze(0)
        result_tokens = torch.cat([result_tokens, next_token])
        tokens = torch.cat([tokens, next_token])
        
        # Stop if end token
        if next_token.item() == model.tokenizer.eos_token_id:
            break
    
    return model.tokenizer.decode(result_tokens)


# Test the manual generation approach
print("="*70)
print("Testing MANUAL token-by-token generation with steering")
print("="*70)
print()

prompt = "I feel neutral about her."
fid = 18699

print("BASELINE:")
baseline = steer_with_feature_manual(prompt, fid, strength=0, max_new_tokens=15)
print(f"Output: {baseline}\n")

print("WITH STEERING:")
steered_pos = steer_with_feature_manual(prompt, fid, strength=50, max_new_tokens=15)
print(f"Output: {steered_pos}\n")

print("OPPOSITE STEERING:")
steered_neg = steer_with_feature_manual(prompt, fid, strength=-50, max_new_tokens=15)
print(f"Output: {steered_neg}\n")

print("\nIf outputs differ, steering is working! 🎉")

Testing MANUAL token-by-token generation with steering

BASELINE:
Output: I feel neutral about her. I don't think she's a bad person. I think she's a

WITH STEERING:
Output: I feel neutral about her.


(?)-??-??-??-??-??

OPPOSITE STEERING:
Output: I feel neutral about her.


She says she's been working with the company for about two


If outputs differ, steering is working! 🎉


In [ ]:
import pandas as pd

TARGET_FID = 107006

release = "gpt2-small-attn-out-v5-128k"
sae_id = "blocks.0.hook_attn_out"
sae = SAE.from_pretrained(release, sae_id)

# ensure lm is available (alias existing LanguageModel instance named `model`)
# some notebooks use `lm` as the language-model wrapper; alias the prepared `model`.
lm = model

# Define prompt sets
positive_prompts = [
    "I like cats",
    "Cats are so cute and fluffy!",
    "Cats are the best pets in the world.",
    "There is nothing better than cats.",
    "Surely you love cats don't you?",
    "Aren't cats just the most wonderful creatures on Earth?",
]

neutral_prompts = [
    "I walked to the store.",
    "The meeting started at noon.",
]

other_negative = [
    "I am in a happy marriage.",
    "I am excited about my family.",
]

def get_sae_features_step1(prompt):
    """Get SAE feature activations for a prompt (using lm from earlier cell)."""
    with lm.trace(prompt):
        h = lm.transformer.h[5].output[0].save()
    
    # h is now a saved tensor, move to device and squeeze batch dimension
    h_device = h.squeeze(0).to(sae.device)
    
    with torch.no_grad():
        feats = sae.encode(h_device)  # [tokens, n_features]
    
    return feats.cpu(), lm.tokenizer.tokenize(prompt)

# Collect activations for feature 18699 across all prompt categories
results_f = {
    "positive": [],
    "neutral": [],
    "other_negative": []
}

print("=" * 70)
print(f"FEATURE {TARGET_FID} ACTIVATION ANALYSIS")
print("=" * 70)

# Process positive prompts
print("\n[POSITIVE PROMPTS]")
for prompt in positive_prompts:
    feats, _ = get_sae_features_step1(prompt)
    max_act = feats[:, TARGET_FID].max().item()
    mean_act = feats[:, TARGET_FID].mean().item()
    results_f["positive"].append({
        "prompt": prompt,
        "max_activation": max_act,
        "mean_activation": mean_act
    })
    print(f"  max={max_act:7.4f}  mean={mean_act:7.4f}  |  {prompt}")

# Process neutral prompts
print("\n[NEUTRAL PROMPTS]")
for prompt in neutral_prompts:
    feats, _ = get_sae_features_step1(prompt)
    max_act = feats[:, TARGET_FID].max().item()
    mean_act = feats[:, TARGET_FID].mean().item()
    results_f["neutral"].append({
        "prompt": prompt,
        "max_activation": max_act,
        "mean_activation": mean_act
    })
    print(f"  max={max_act:7.4f}  mean={mean_act:7.4f}  |  {prompt}")

# Process other-negative prompts
print("\n[OTHER NEGATIVE PROMPTS]")
for prompt in other_negative:
    feats, _ = get_sae_features_step1(prompt)
    max_act = feats[:, TARGET_FID].max().item()
    mean_act = feats[:, TARGET_FID].mean().item()
    results_f["other_negative"].append({
        "prompt": prompt,
        "max_activation": max_act,
        "mean_activation": mean_act
    })
    print(f"  max={max_act:7.4f}  mean={mean_act:7.4f}  |  {prompt}")

# Compute statistics
print("\n" + "=" * 70)
print("STATISTICS")
print("=" * 70)

for category in ["positive", "neutral", "other_negative"]:
    max_acts = [r["max_activation"] for r in results_f[category]]
    mean_acts = [r["mean_activation"] for r in results_f[category]]
    
    print(f"\n{category.upper()}:")
    print(f"  Mean of max activations:  {sum(max_acts) / len(max_acts):.4f}")
    print(f"  Mean of mean activations: {sum(mean_acts) / len(mean_acts):.4f}")
    print(f"  Std of max activations:   {torch.tensor(max_acts).std().item():.4f}")

# Compare across categories
positive_max_mean = sum([r["max_activation"] for r in results_f["positive"]]) / len(results_f["positive"])
neutral_max_mean = sum([r["max_activation"] for r in results_f["neutral"]]) / len(results_f["neutral"])
other_max_mean = sum([r["max_activation"] for r in results_f["other_negative"]]) / len(results_f["other_negative"])

print("\n" + "-" * 70)
print("COMPARISON: Mean of Max Activations")
print("-" * 70)
print(f"  Postive:      {positive_max_mean:.4f}")
print(f"  Neutral:       {neutral_max_mean:.4f}")
print(f"  Other-Neg:     {other_max_mean:.4f}")
print(f"\n  Postive vs Neutral:       +{(positive_max_mean - neutral_max_mean):.4f}")
print(f"  Postive vs Other-Neg:     +{(positive_max_mean - other_max_mean):.4f}")

if positive_max_mean > neutral_max_mean and positive_max_mean > other_max_mean:
    print(f"\n✅ Feature {TARGET_FID} preferentially activates this feature!")
else:
    print(f"\n⚠️  Feature {TARGET_FID} does not strongly prefer this feature.")

FEATURE 4 ACTIVATION ANALYSIS

[POSITIVE PROMPTS]
  max= 0.0000  mean= 0.0000  |  I like cats
  max= 0.0000  mean= 0.0000  |  Cats are so cute and fluffy!
  max= 0.0000  mean= 0.0000  |  Cats are the best pets in the world.
  max= 0.0000  mean= 0.0000  |  There is nothing better than cats.
  max= 0.0000  mean= 0.0000  |  Surely you love cats don't you?
  max= 0.0000  mean= 0.0000  |  Aren't cats just the most wonderful creatures on Earth?

[NEUTRAL PROMPTS]
  max= 0.0000  mean= 0.0000  |  I walked to the store.
  max= 0.0000  mean= 0.0000  |  The meeting started at noon.

[OTHER NEGATIVE PROMPTS]
  max= 0.0000  mean= 0.0000  |  I am in a happy marriage.
  max= 0.0000  mean= 0.0000  |  I am excited about my family.

STATISTICS

POSITIVE:
  Mean of max activations:  0.0000
  Mean of mean activations: 0.0000
  Std of max activations:   0.0000

NEUTRAL:
  Mean of max activations:  0.0000
  Mean of mean activations: 0.0000
  Std of max activations:   0.0000

OTHER_NEGATIVE:
  Mean of max ac

In [13]:
# Step 2: Feature Discovery via Activation Contrast
# Discover which features activate most specifically on jealousy prompts

print("\n" + "=" * 70)
print("FEATURE DISCOVERY: Which features activate on our feature?")
print("=" * 70)

# Collect max activations for all features across all prompts
all_prompts_positive = {
    "positive": positive_prompts,
    "neutral": neutral_prompts,
    "other_negative": other_negative
}

# Store activations per category: [n_prompts, n_features]
activations_per_category = {}

for category, prompts in all_prompts_positive.items():
    category_acts = []
    for prompt in prompts:
        feats, _ = get_sae_features_step1(prompt)  # Fixed: was get_sae_features
        # Take max activation across tokens for each feature
        max_acts = feats.max(dim=0).values  # [n_features]
        category_acts.append(max_acts)
    activations_per_category[category] = torch.stack(category_acts)  # [n_prompts, n_features]

print(f"\nActivations shape per category:")
for cat, acts in activations_per_category.items():
    print(f"  {cat:15s}: {acts.shape}")

# Compute mean activation per category
mean_acts_positive = activations_per_category["positive"].mean(dim=0)  # [n_features]
mean_acts_neutral = activations_per_category["neutral"].mean(dim=0)
mean_acts_other = activations_per_category["other_negative"].mean(dim=0)

print(f"\n✓  Computed mean activations across categories")

# Define specificity score:
# A feature is specific to jealousy if it activates strongly on jealousy
# but weakly on neutral and other-negative prompts
# specificity = mean_jealousy - max(mean_neutral, mean_other_neg)

baseline = torch.maximum(mean_acts_neutral, mean_acts_other)
specificity = mean_acts_positive - baseline  # [n_features]

# Get top features by specificity
top_k = 25
top_feature_indices = torch.argsort(specificity, descending=True)[:top_k]

print(f"\nTop {top_k} features by jealousy specificity:")
print(f"{'Rank':<6} {'Feature':>8} {'Specificity':>12} {'J_mean':>10} {'N_mean':>10} {'O_mean':>10}")
print("-" * 65)

for rank, fidx in enumerate(top_feature_indices, 1):
    fidx = fidx.item()
    spec = specificity[fidx].item()
    j_mean = mean_acts_positive[fidx].item()
    n_mean = mean_acts_neutral[fidx].item()
    o_mean = mean_acts_other[fidx].item()
    
    marker = " ◀ TARGET!" if fidx == 18699 else ""
    print(f"{rank:<6} {fidx:>8} {spec:>12.4f} {j_mean:>10.4f} {n_mean:>10.4f} {o_mean:>10.4f}{marker}")

# Check where feature 18699 ranks
# if 18699 in top_feature_indices:
#     rank_18699 = (top_feature_indices == 18699).nonzero(as_tuple=True)[0].item() + 1
#     print(f"\n✅ Feature 18699 is in top-{top_k} at rank #{rank_18699}!")
# else:
#     all_sorted = torch.argsort(specificity, descending=True)
#     rank_18699 = (all_sorted == 18699).nonzero(as_tuple=True)[0].item() + 1
#     spec_18699 = specificity[18699].item()
#     print(f"\n⚠️  Feature 18699 ranked #{rank_18699} (specificity={spec_18699:.4f})")

print(f"\n" + "=" * 70)
print("Analysis complete! Features ranked by jealousy-specificity.")
print("=" * 70)


FEATURE DISCOVERY: Which features activate on our feature?

Activations shape per category:
  positive       : torch.Size([6, 131072])
  neutral        : torch.Size([2, 131072])
  other_negative : torch.Size([2, 131072])

✓  Computed mean activations across categories

Top 25 features by jealousy specificity:
Rank    Feature  Specificity     J_mean     N_mean     O_mean
-----------------------------------------------------------------
1         77740      13.1231    13.1231     0.0000     0.0000
2         18858      11.2721    17.1955     5.0364     5.9234
3         36389       9.4684     9.4684     0.0000     0.0000
4        107006       8.3154     8.3154     0.0000     0.0000
5         13272       7.7394     7.7394     0.0000     0.0000
6         45591       7.5391     7.5391     0.0000     0.0000
7        119199       6.6493     6.6493     0.0000     0.0000
8         33790       6.5063     6.5063     0.0000     0.0000
9          8290       6.2240     6.2240     0.0000     0.0000
10

In [10]:
# Blind top-5 causal test: can any of the strongest features push a neutral prompt toward cats?
# This does not assume a known target feature. It scores generated text for cat-related language.

#probe_prompt = "I saw a "
probe_prompt = "In the Zoo i saw a animal. It "
boost_strength = 50.0
max_new_tokens = 25

if "top_feature_indices" in globals():
    candidate_features = top_feature_indices[:5].tolist()
else:
    candidate_features = torch.argsort(specificity, descending=True)[:5].tolist()

cat_words = [
    "cat", "cats", "kitten", "kittens", "kitty", "kitties", "feline", "felines",
    "purr", "purring", "meow", "meowing", "whisker", "whiskers", "tabby"
]

def cat_score(text):
    lowered = text.lower()
    return sum(word in lowered for word in cat_words)


def generate_with_feature(prompt, feature_id, strength=0.0, steps=25):
    if "steer_with_feature_manual" in globals():
        return steer_with_feature_manual(prompt, feature_id, strength=strength, max_new_tokens=steps)

    delta = strength * sae.W_dec[feature_id].detach()
    tokens = model.tokenizer.encode(prompt, return_tensors="pt")[0]
    result_tokens = tokens.clone()

    for _ in range(steps):
        with model.trace(model.tokenizer.decode(tokens)):
            resid = model.transformer.h[5].output[0]
            resid[:] = resid + delta.unsqueeze(0).unsqueeze(0)
            logits = model.lm_head.output.save()

        next_token = torch.argmax(logits[0, -1, :]).unsqueeze(0)
        result_tokens = torch.cat([result_tokens, next_token])
        tokens = torch.cat([tokens, next_token])
        if next_token.item() == model.tokenizer.eos_token_id:
            break

    return model.tokenizer.decode(result_tokens)

print("=" * 70)
print("BLIND TOP-5 CAT STEERING TEST")
print("=" * 70)
print(f"Probe prompt: {probe_prompt!r}")
print(f"Boost strength: {boost_strength}")
print(f"Max new tokens: {max_new_tokens}")
print()

baseline = generate_with_feature(probe_prompt, candidate_features[0], strength=0.0, steps=max_new_tokens)
baseline_score = cat_score(baseline)
print("BASELINE OUTPUT:")
print(baseline)
print(f"cat_score={baseline_score}")
print()

results = []
for fid in candidate_features:
    output = generate_with_feature(probe_prompt, fid, strength=boost_strength, steps=max_new_tokens)
    score = cat_score(output)
    results.append((fid, score, output))
    print(f"feature {fid:6d} | cat_score={score:2d}")
    print(output)
    print("-" * 70)

best = max(results, key=lambda item: item[1])
print()
print(f"Best cat score: feature {best[0]} with cat_score={best[1]}")
if best[1] > baseline_score:
    print("At least one top-5 feature increased cat-related language relative to baseline.")
else:
    print("None of the top-5 features clearly shifted the generation toward cats on this probe.")

BLIND TOP-5 CAT STEERING TEST
Probe prompt: 'In the Zoo i saw a animal. It '
Boost strength: 50.0
Max new tokens: 25

BASELINE OUTPUT:
In the Zoo i saw a animal. It   was a black bear. It was a black bear. It was a black bear. It was a black bear. It
cat_score=0

feature  77740 | cat_score= 1
In the Zoo i saw a animal. It   was a cat.
I was looking at the cat.
I was looking at the cat.
I was looking
----------------------------------------------------------------------
feature  18858 | cat_score= 0
In the Zoo i saw a animal. It   was a little girl. She was a little girl who was very young. She was very young. She was very young
----------------------------------------------------------------------
feature  36389 | cat_score= 0
In the Zoo i saw a animal. It   was a very special gift to have a gift like this.
 I have a lot of items that I have in my
----------------------------------------------------------------------
feature 107006 | cat_score= 1
In the Zoo i saw a animal. It   was ver